In [ ]:
import sage.all
import numpy as np
import scipy

In [ ]:
distances_ellipse_full= {
    "Mercure": (0.307, 0.466), 
    "Vénus": (0.718, 0.728),
    "Terre": (0.983, 1.017),
    "Mars": (1.381, 1.666),
    "Jupiter": (4.950, 5.457),
    "Saturne": (9.027, 10.124),
    "Uranus": (18.289, 20.096),
    "Neptune": (29.809, 30.334),
}
distances_ellipse = {
    "Mercure": (0.307, 0.466), 
    "Vénus": (0.718, 0.728),
    "Terre": (0.983, 1.017),
    "Mars": (1.381, 1.666),
}

In [ ]:
planetes = {
    "Mercure": {"distance": 0.4, "period": 88, "radius": 0.06, "color": "gray", "inclinaison": 7 * np.pi / 180, "Demi_g_axe_km": 57909050, "Périphélie_km": 46001200, "Excentricité": 0.2056},
    "Venus": {"distance": 0.7, "period": 225, "radius": 0.09, "color": "yellow", "inclinaison": 3.4 * np.pi / 180, "Demi_g_axe_km": 108209500, "Périphélie_km": 107476000, "Excentricité": 0.0068},
    "Terre": {"distance": 1.0, "period": 365, "radius": 0.15, "color": "blue", "inclinaison": 0, "Demi_g_axe_km": 149597887.5, "Périphélie_km": 147098074, "Excentricité": 0.0167},
    "Mars": {"distance": 1.5, "period": 687, "radius": 0.09, "color": "red", "inclinaison": 1.9 * np.pi / 180, "Demi_g_axe_km": 227944000, "Périphélie_km": 206655000, "Excentricité": 0.0934},
    "Jupiter": {"distance": 5.2, "period": 4333, "radius": 0.26, "color": "orange", "inclinaison": 1.3 * np.pi / 180, "Demi_g_axe_km": 778300000, "Périphélie_km": 740560000, "Excentricité": 0.0489},
    "Saturne": {"distance": 9.5, "period": 10759, "radius": 0.22, "color": "gold", "inclinaison": 2.5 * np.pi / 180, "Demi_g_axe_km": 1429400000, "Périphélie_km": 1349467000, "Excentricité": 0.0565},
    "Uranus": {"distance": 19.8, "period": 30687, "radius": 0.17, "color": "lightblue", "inclinaison": 0.8 * np.pi / 180, "Demi_g_axe_km": 2875000000, "Périphélie_km": 2734000000, "Excentricité": 0.0465},
    "Neptune": {"distance": 30.1, "period": 60190, "radius": 0.17, "color": "darkblue", "inclinaison": 1.8 * np.pi / 180, "Demi_g_axe_km": 4504500000, "Périphélie_km": 4459700000, "Excentricité": 0.0086}
}

In [ ]:
ua = 149597887.5
for b in planetes.keys():
    planetes[b]["C"] =  planetes[b]["Demi_g_axe_km"] - planetes[b]["Périphélie_km"]
    planetes[b]["Demi_g_axe_ua"] =  planetes[b]["Demi_g_axe_km"]/ua
    planetes[b]["C_ua"] =  planetes[b]["C"]/ua
    planetes[b]["Demi_p_axe_ua"] =  sqrt(planetes[b]["Demi_g_axe_ua"]**2 - planetes[b]["C_ua"]**2)

In [ ]:
i = 6
Lune = ("Terre",0.1,27.32,2*pi/27.32, 45.145 * math.pi / 180)

In [ ]:
v0 = 4

In [ ]:
theta_f = 45 * pi / 180 #Inclinaison de la fusée par rapport à la Terre
theta_cos_f = cos(theta_f)
theta_sin_f = sin(theta_f)
phi = 90 * pi / 180 #Cap d'inclinaison de la fusée (vers l'est)
phi_cos = cos(phi)
phi_sin = sin(phi)

In [ ]:
def position_ellipse_geometrique(planet_name, t):
    a = planetes[planet_name]["Demi_g_axe_km"]
    e = planetes[planet_name]["Excentricité"]
    b = a * sqrt(1 - e**2)
    T = planetes[planet_name]["period"]

    theta = 2 * pi * t / T

    x = a * cos(theta)
    y = b * sin(theta)
    return x, y

In [ ]:
def Kepler_equation(E, planet_name, M):
    e = planetes[planet_name]["Excentricité"]
    return E - e * sin(E) - M

def approximativ_position(planet_name, time):
    period = float(planetes[planet_name]["period"])
    average_anomaly = float(2 * pi * time / period)
    eccentric_anomaly = scipy.optimize.newton(
    Kepler_equation,
    average_anomaly,
    args=(planet_name, average_anomaly)
    )
    a = planetes[planet_name]["Demi_g_axe_km"]
    e = planetes[planet_name]["Excentricité"]
    x = a * (cos(eccentric_anomaly) - e)
    y = a * sqrt(1 - e**2) * sin(eccentric_anomaly)
    return x, y

### FONCTION AVEC ELLIPSES

In [ ]:
def dessine_planetes_ellipse_3d(k):
    ECHELLE = 1e7  # Pour rendre le tout visible à une échelle "humaine"
    S = sphere(center=(0, 0, 0), size = int(2 * i), opacity = 0)
    S += sphere(center=(0, 0, 0), size=0.3 * i, fill=True, color="yellow")  # Dessiner l'étoile au centre
    for nom_planete, dic in planetes.items():
            
        a = dic["Demi_g_axe_km"]
        b = a * sqrt(1 - dic["Excentricité"]**2)
        inclinaison = dic["inclinaison"]
        couleur = dic["color"]
        rayon = dic["radius"]

        # Position de Jupiter
        x, y = position_ellipse_geometrique(nom_planete, k)
        z = sin(inclinaison) * 0.1  # faible élévation

        # Réduction d’échelle des positions
        x /= ECHELLE
        y /= ECHELLE
        z /= ECHELLE

        # Tracé de l'ellipse (réduite aussi)
        theta = np.linspace(0, 2 * np.pi, 300)
        ellipse_x = a * cos(theta) / ECHELLE
        ellipse_y0 = b * sin(theta) / ECHELLE
        ellipse_z0 = np.zeros_like(theta)

        # Inclinaison appliquée
        ellipse_y = [yy * cos(inclinaison) for yy in ellipse_y0]
        ellipse_z = [yy * sin(inclinaison) for yy in ellipse_y0]

        # Ajouter les points 3D de l'ellipse dans l'objet S
        for r in range(len(theta)):
            S += point3d((ellipse_x[r], ellipse_y[r], ellipse_z[r]), size=0.1, color='black')        
        
        # Dessin de la planète (rayon aussi ajusté pour visibilité)
        S += sphere(center=(x, y, z), size=rayon * 10, fill=True, color=couleur)
        
    return S

# Créer l'animation
def animation_systeme_solaire():
    liste = [dessine_planetes_ellipse_3d(k) for k in range(1, 10)]
    return animate(liste, figsize=(25, 25))

### FONCTION SANS LES ELLIPSES

In [ ]:
def dessine_planetes_ellipse_3d_v2(k):
    S = sphere(center=(0, 0, 0), size = 2 * i, opacity = 0)
    S += sphere(center=(0, 0, 0), size=0.3 * i, fill=True, color="yellow")  # Dessiner l'étoile au centre
    for nom_planete, dic in planetes.items():
            
        distance = dic["distance"] * i * 3
        periode = dic["period"]
        couleur = dic["color"]
        rayon = dic["radius"]
        inclinaison = dic["inclinaison"]

        angle = 2 * pi * 3 * k / periode #Avec 10 pour augmenter la vitesse de nos astres
        x = distance * cos(angle)
        y = distance * sin(angle)
            
        z = sin(inclinaison) * 0.1  # Légère variation sur l'axe z pour rendre l'orbite moins plate
            
        S += circle(center=(0, 0), radius=distance, fill=False, color='black')
        S += sphere(center=(x, y, z), size=rayon * i, fill=True, color=couleur)
        
        if nom_planete == "Terre" :
            xt = distance * cos(angle)
            yt = distance * sin(angle)
            angle_t = angle
            rayont = rayon
            xs = xt + 0.1 * i * 3 * cos(3*k*2*pi/(27.32))
            ys = yt + 0.1 * i * 3 * sin(3*k*2*pi/(27.32))
            zs = sin(Lune[4]) * 0.1
            xf = xt + v0 * cos(angle) * k / 60
            yf = yt + v0 * sin(angle) * k / 60
            zf = v0 * z
                
    S += sphere(center=(xt,yt,zs * 0.1), size=0.1 * i * 3 + 0.05, fill = False, color = "black", opacity=0.1)
    S += sphere(center=(xs,ys,zs), size=0.02 * i, fill = True, color = "red")
    S += sphere((xf, yf, zf), size=0.2, fill=True, color='red')

    return S

# Créer l'animation
def animation_systeme_solaire_v2():
    liste = [dessine_planetes_ellipse_3d_v2(k) for k in range(1, 131)]
    return animate(liste, figsize=(25, 25))

### FONCTION AVEC ELLIPSE POUR JUPITER SEULEMENT

In [ ]:
def dessine_planetes_ellipse_3d_jupiter():
    ECHELLE = 1e7  # Pour rendre le tout visible à une échelle "humaine"
    liste = []
    
    for k in range(1, 50):
        S = sphere(center=(0, 0, 0), size=int(2 * i), opacity=0)
        S += sphere(center=(0, 0, 0), size=0.3 * i, fill=True, color="yellow")  # Soleil

        # Infos de Jupiter
        dic = planetes["Jupiter"]
        a = dic["Demi_g_axe_km"]
        b = a * sqrt(1 - dic["Excentricité"]**2)
        inclinaison = dic["inclinaison"]
        couleur = dic["color"]
        rayon = dic["radius"]

        # Position de Jupiter
        x, y = position_ellipse_geometrique("Jupiter", k)
        z = sin(inclinaison) * 0.1  # faible élévation

        # Réduction d’échelle des positions
        x /= ECHELLE
        y /= ECHELLE
        z /= ECHELLE

        # Tracé de l'ellipse (réduite aussi)
        theta = np.linspace(0, 2 * np.pi, 300)
        ellipse_x = a * cos(theta) / ECHELLE
        ellipse_y0 = b * sin(theta) / ECHELLE
        ellipse_z0 = np.zeros_like(theta)

        # Inclinaison appliquée
        ellipse_y = [yy * cos(inclinaison) for yy in ellipse_y0]
        ellipse_z = [yy * sin(inclinaison) for yy in ellipse_y0]

        # Ajouter les points 3D de l'ellipse dans l'objet S
        for r in range(len(theta)):
            S += point3d((ellipse_x[r], ellipse_y[r], ellipse_z[r]), size=0.1, color='black')        
        
        # Dessin de la planète (rayon aussi ajusté pour visibilité)
        S += sphere(center=(x, y, z), size=rayon * 10, fill=True, color=couleur)

        liste.append(S)

    return liste

# Créer l'animation
def animation_systeme_solaire_jupiter():
    return animate(dessine_planetes_ellipse_3d_jupiter(), figsize=(25, 25))

### ANIMATION SANS LES ELLIPSES

In [ ]:
a = animation_systeme_solaire_v2()

In [ ]:
a.interactive(delay=10, iterations=2, viewpoint=((0.215, 0.3686, 0.9044), 124.36))

### ANIMATION POUR LES ELLIPSES

In [ ]:
b = animation_systeme_solaire()

In [ ]:
b.interactive(delay=10, iterations=2, viewpoint=((0.215, 0.3686, 0.9044), 124.36))

### ANIMATION AVEC ELLIPSE POUR JUPITER

In [ ]:
c = animation_systeme_solaire_jupiter()

In [ ]:
c.interactive(delay=10, iterations=2, viewpoint=((0.215, 0.3686, 0.9044), 124.36))